# 00 · Configuración de Colab + GitHub
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Valoo-1201/vfc-caos-som/blob/main/notebooks/00_setup_colab_github.ipynb)

Este notebook se usa **una vez por persona** para comprobar que todo funciona, y luego cada vez que quieras subir cambios.

**Antes de empezar**
1. Crea un token de GitHub: *Settings → Developer settings → Fine-grained tokens*, solo para este repositorio, permiso **Contents: Read and write**.
2. En Colab abre el panel 🔑 **Secretos**, crea `GITHUB_TOKEN` con ese valor y activa el acceso para este notebook.
3. Cambia `REPO` en la celda 0 por `usuario/vfc-caos-som`.

In [ ]:
#@title 0 · Configuración (ejecutar siempre primero)
REPO = "Valoo-1201/vfc-caos-som"   # <- usuario/repositorio en GitHub
RAMA = "main"
USAR_DRIVE = True                  # guarda las series R-R en Google Drive para no descargarlas otra vez
MODO_DEMO = False                  # True = datos sintéticos (solo para probar el código sin internet)

import os, sys, subprocess
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    os.chdir("/content")
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")      # Colab -> 🔑 Secretos (solo hace falta si el repo es privado)
    except Exception:
        pass
    url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
    if not os.path.exists("/content/vfc-caos-som"):
        subprocess.run(["git", "clone", "-q", "-b", RAMA, url, "/content/vfc-caos-som"], check=True)
    else:
        subprocess.run(["git", "-C", "/content/vfc-caos-som", "pull", "-q"], check=False)
    os.chdir("/content/vfc-caos-som")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    if USAR_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        DATA_DIR = "/content/drive/MyDrive/vfc-caos-som-data"
    else:
        DATA_DIR = "/content/data"
else:  # ejecución local: el notebook está en notebooks/
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")
    DATA_DIR = "data"

sys.path[:0] = [os.path.abspath("src"), os.getcwd()]
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs("results/figures", exist_ok=True)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
SEED = 42
np.random.seed(SEED)
pd.set_option("display.precision", 3)
print("Repositorio:", os.getcwd(), "| Datos:", DATA_DIR, "| Demo:", MODO_DEMO)

## Comprobación rápida
Ejecuta las pruebas de las métricas: todas deben decir `OK`.

In [ ]:
import importlib, time
import tests.test_metrics as T
importlib.reload(T)
for nombre in [n for n in dir(T) if n.startswith("test_")]:
    t0 = time.time()
    try:
        getattr(T, nombre)(); print("OK  ", nombre, f"{time.time()-t0:.1f}s")
    except AssertionError as e:
        print("FALLA", nombre, e)

## Subir tus cambios a GitHub
Trabaja siempre en **tu propia rama** y abre un *pull request* en GitHub para que otra compañera lo revise.

> Antes de subir un notebook: *Entorno de ejecución → Reiniciar y ejecutar todo*. Si solo cambiaste un notebook, también puedes usar *Archivo → Guardar una copia en GitHub*.

In [ ]:
#@title Subir cambios (git add / commit / push)
NOMBRE = "Tu Nombre"            #@param {type:"string"}
EMAIL = "tu_correo@ejemplo.com" #@param {type:"string"}
MI_RAMA = "feat/mi-tarea"       #@param {type:"string"}
MENSAJE = "Describe el cambio"  #@param {type:"string"}

def git(*args, check=True):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    print((r.stdout + r.stderr).strip())
    if check and r.returncode:
        raise RuntimeError("git " + " ".join(args))

git("config", "user.name", NOMBRE)
git("config", "user.email", EMAIL)
git("checkout", "-B", MI_RAMA)
git("add", "-A")
git("commit", "-m", MENSAJE, check=False)   # no falla si no hay cambios
git("push", "-u", "origin", MI_RAMA)
print(f"Listo. Abre el pull request en https://github.com/{REPO}/compare/{MI_RAMA}")